# TomeSlicer — Marker Extraction (Stage 1)

Offloads heavy Marker (Surya OCR/layout) inference to a Colab T4 GPU.
Outputs per-page JSON files that `tomeslicer.py` can consume locally for Stage 2 (Gemini enrichment).

**Runtime:** Select **T4 GPU** under Runtime → Change runtime type.

In [ ]:
!pip install -q marker-pdf

In [ ]:
from google.colab import userdata
import os

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
PDF_PATH = "/content/drive/MyDrive/dnd/book.pdf"
OUTPUT_DIR = "/content/drive/MyDrive/dnd/output"
PAGE_LIMIT = None  # Set to int to limit pages

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Go to Runtime → Change runtime type → T4 GPU.")

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_mem / (1024**3)
print(f"GPU: {gpu_name} ({vram_gb:.1f} GB VRAM)")

os.environ["TORCH_DEVICE"] = "cuda"
os.environ["INFERENCE_RAM"] = "16"
os.environ["VRAM_PER_TASK"] = "5"

In [ ]:
import re
from html import unescape

MARKER_TYPE_MAP = {
    "SectionHeader": "section_heading",
    "Text": "body_text",
    "TextInlineMath": "body_text",
    "Table": "table",
    "Figure": "illustration",
    "Picture": "illustration",
    "Caption": "body_text",
    "Code": "body_text",
    "Equation": "rule_callout",
    "Form": "table",
    "Footnote": "page_footer",
    "PageFooter": "page_footer",
    "PageHeader": "page_footer",
    "ListGroup": "body_text",
    "ListItem": "body_text",
    "HandwrittenText": "body_text",
}


def html_to_text(html: str) -> str:
    """Convert Marker HTML output to markdown-formatted text."""
    text = html
    for i in range(1, 7):
        text = re.sub(
            rf"<h{i}[^>]*>(.*?)</h{i}>",
            rf"{'#' * i} \1\n",
            text,
            flags=re.DOTALL,
        )
    text = re.sub(r"<(b|strong)[^>]*>(.*?)</\1>", r"**\2**", text, flags=re.DOTALL)
    text = re.sub(r"<(i|em)[^>]*>(.*?)</\1>", r"*\2*", text, flags=re.DOTALL)
    text = re.sub(r"<li[^>]*>(.*?)</li>", r"- \1\n", text, flags=re.DOTALL)
    text = re.sub(r"</?[ou]l[^>]*>", "\n", text)
    text = re.sub(r"<br\s*/?>", "\n", text)
    text = re.sub(r"</p>", "\n\n", text)
    text = re.sub(r"<p[^>]*>", "", text)
    text = re.sub(r"<[^>]+>", "", text)
    text = unescape(text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def html_table_to_markdown(html: str) -> str:
    """Convert HTML table to GitHub Flavored Markdown table."""
    rows = re.findall(r"<tr[^>]*>(.*?)</tr>", html, flags=re.DOTALL)
    if not rows:
        return html_to_text(html)
    md_rows = []
    for row in rows:
        cells = re.findall(r"<t[hd][^>]*>(.*?)</t[hd]>", row, flags=re.DOTALL)
        cells = [re.sub(r"<[^>]+>", "", c).strip() for c in cells]
        cells = [unescape(c) for c in cells]
        md_rows.append("| " + " | ".join(cells) + " |")
    if md_rows:
        num_cols = md_rows[0].count("|") - 1
        separator = "| " + " | ".join(["---"] * max(num_cols, 1)) + " |"
        md_rows.insert(1, separator)
    return "\n".join(md_rows)


def convert_marker_block(block, page_width: float, page_height: float) -> dict:
    """Convert a Marker JSON block to TomeSlicer intermediate format."""
    from marker.output import json_to_html

    polygon = block.polygon
    xs = [p[0] for p in polygon]
    ys = [p[1] for p in polygon]
    x_min, x_max = min(xs), max(xs)
    y_min, y_max = min(ys), max(ys)

    box_2d = [
        max(0, min(1000, int(y_min / page_height * 1000))),
        max(0, min(1000, int(x_min / page_width * 1000))),
        max(0, min(1000, int(y_max / page_height * 1000))),
        max(0, min(1000, int(x_max / page_width * 1000))),
    ]
    if box_2d[0] >= box_2d[2]:
        box_2d[2] = min(1000, box_2d[0] + 1)
    if box_2d[1] >= box_2d[3]:
        box_2d[3] = min(1000, box_2d[1] + 1)

    bt = block.block_type
    block_type_str = bt.name if hasattr(bt, "name") else str(bt).split(".")[-1]

    full_html = json_to_html(block)
    if block_type_str == "Table":
        text = html_table_to_markdown(full_html)
    else:
        text = html_to_text(full_html)

    preliminary_type = MARKER_TYPE_MAP.get(block_type_str, "body_text")

    return {
        "text": text,
        "box_2d": box_2d,
        "html": full_html,
        "preliminary_type": preliminary_type,
        "marker_block_type": block_type_str,
    }

In [ ]:
import json
import time
from pathlib import Path

from marker.converters.pdf import PdfConverter
from marker.models import create_model_dict
from marker.config.parser import ConfigParser

start_time = time.time()

print("Loading Marker models...")
models = create_model_dict()

config = {
    "output_format": "json",
    "use_llm": True,
    "gemini_api_key": os.environ["GOOGLE_API_KEY"],
}
if PAGE_LIMIT:
    config["page_range"] = ",".join(str(p) for p in range(PAGE_LIMIT))

config_parser = ConfigParser(config)
converter = PdfConverter(
    config=config_parser.generate_config_dict(),
    artifact_dict=models,
    processor_list=config_parser.get_processors(),
    renderer=config_parser.get_renderer(),
    llm_service=config_parser.get_llm_service(),
)

print(f"Running Marker on {PDF_PATH}...")
rendered = converter(PDF_PATH)

output_path = Path(OUTPUT_DIR)
output_path.mkdir(parents=True, exist_ok=True)

total_blocks = 0
for page_idx, page in enumerate(rendered.children):
    bbox = page.bbox
    page_width = bbox[2] - bbox[0]
    page_height = bbox[3] - bbox[1]

    blocks = []
    for child in page.children:
        blocks.append(convert_marker_block(child, page_width, page_height))

    page_number = page_idx + 1
    page_data = {
        "page_number": page_number,
        "blocks": blocks,
    }

    out_file = output_path / f"page-{page_number:03d}.json"
    out_file.write_text(json.dumps(page_data, indent=2, ensure_ascii=False))
    total_blocks += len(blocks)
    print(f"  page-{page_number:03d}.json — {len(blocks)} blocks")

elapsed = time.time() - start_time
print(f"\nDone. Wrote {len(rendered.children)} pages to {OUTPUT_DIR}")

In [ ]:
pages_processed = len(rendered.children)
minutes, seconds = divmod(elapsed, 60)

print(f"Pages processed: {pages_processed}")
print(f"Total blocks:    {total_blocks}")
print(f"Elapsed time:    {int(minutes)}m {seconds:.1f}s")
print(f"Avg per page:    {elapsed / max(pages_processed, 1):.1f}s")
print(f"Output dir:      {OUTPUT_DIR}")